In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


## Tutorial on how to convert a Topologic Graph to a GraphViz Graph
### You must first correctly and separately install GraphViz

In [ ]:
# Import TopologicPy modules. This is not needed on other computers
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## Import the needed libraries
### Make sure you have the latest version of TopologicPy (0.8.24 or newer)

In [ ]:
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph
from topologicpy.Helper import Helper
print("This jupyter notebook requires topologicpy v.0.8.24 or newer")
print(Helper.Version())

## Create a sample CellComplex and derive its dual graph

In [ ]:
cc = CellComplex.Prism(uSides=3)
cells = Topology.Cells(cc)
for i, cell in enumerate(cells):
    d = Dictionary.ByKeysValues(["vertexLabel","vertexShape","vertexFillColor"], [f"Cell {i+1}", "circle", "yellow"])
    cell = Topology.SetDictionary(cell, d)

faces = Topology.Faces(_as_topology(cc))
for i, face in enumerate(faces):
    d = Dictionary.ByKeysValues(["vertexLabel","vertexShape","vertexFillColor"], [f"Face {i+1}", "box", "lightgrey"])
    face = Topology.SetDictionary(face, d)
g = TGraph.ByTopology(cc, toExteriorTopologies=True)
graph_edges = _edge_records(g)
colors = ["red", "blue", "green", "cyan", "magenta", "yellow", "black", "brown", "orange"]
widths = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
for e in graph_edges:
    d = _record_dictionary(e)
    category = Dictionary.ValueAtKey(d, "category")
    edge_color = colors[category]
    edge_width = widths[category]
    d = Dictionary.SetValuesAtKeys(d, ["edgeColor", "edgeWidth"], [edge_color, edge_width])
    e = Topology.SetDictionary(e, d)
Topology.Show(cc, g)

## Convert the topologic graph to a GraphViz Graph

In [ ]:
gv_graph = TGraph.GraphVizGraph(g, showVertexLabel=True, vertexLabelKey="vertexLabel")
# Configure the returned graphviz.Digraph using the Graphviz API.


## Display the GraphViz Graph

In [ ]:
gv_graph

## Export the dual graph to a GraphViz .gv file
### Make sure you change the path first

In [ ]:
status = TGraph.ExportToGraphVizGraph(g, path=path, showVertexLabel=True, vertexLabelKey="vertexLabel", overwrite=True)


In [ ]:
eigenvector_centralities = TGraph.EigenVectorCentrality(g, normalize=True, mantissa=2)
print(sum(eigenvector_centralities)/len(eigenvector_centralities))

pr_centralities = TGraph.PageRank(g, mantissa=2)
print(sum(pr_centralities)/len(pr_centralities))
TGraph.Show(g, vertexColorKey="evc_color")
TGraph.Show(g, vertexColorKey="pr_color")

comparison = TGraph.Compare(g, g, weightEigenVectorCentrality=1, weightPageRank=1)
print(comparison["eigenvector_centrality"], comparison["pagerank"], comparison["overall"])

In [ ]:
def ConvexHull(topology, mantissa: int = 6, tolerance: float = 0.0001, silent: bool = False):
    """
    Computes the convex hull of the input topology.
    Returns a Face in 2D (even embedded in 3D), or a Cell/Shell in 3D.

    Parameters
    ----------
    topology : topologic_core.Topology
        The input Topology.
    mantissa : int, optional
        Precision of the coordinates. Default is 6.
    tolerance : float, optional
        Tolerance value for geometric operations. Default is 0.0001.
    silent : bool, optional
        If True, suppresses warning and error messages.

    Returns
    -------
    topologic_core.Topology
        The convex hull of the topology as a Face (2D) or Cell/Shell (3D).
    """
    from topologicpy.Vertex import Vertex
    from topologicpy.Edge import Edge
    from topologicpy.Wire import Wire
    from topologicpy.Face import Face
    from topologicpy.Cell import Cell
    from topologicpy.Shell import Shell
    from topologicpy.Cluster import Cluster
    from topologicpy.Topology import Topology

    import numpy as np
    import math
    from scipy.spatial import ConvexHull as SciPyConvexHull
    from numpy.linalg import svd

    if not Topology.IsInstance(_as_topology(topology), 'Topology'):
        if not silent:
            print("Topology.ConvexHull - Error: Input is not a valid topology. Returning None.")
        return None

    vertices = Topology.Vertices(_as_topology(topology))
    coords = np.array([Vertex.Coordinates(_as_topology(v), mantissa=mantissa) for v in vertices])

    if coords.shape[0] < 3:
        if not silent:
            print("Topology.ConvexHull - Error: Need at least 3 points to compute a convex hull.")
        return None

    # Check if points are coplanar using SVD
    _, s, vh = svd(coords - coords.mean(axis=0))
    rank = (s > tolerance).sum()

    if rank <= 2:
        # Project to 2D in the best-fit plane
        normal = vh[2, :]
        x_axis = vh[0, :]
        y_axis = vh[1, :]
        origin = coords.mean(axis=0)

        def project(p):
            rel = p - origin
            return [np.dot(rel, x_axis), np.dot(rel, y_axis)]

        coords_2d = np.array([project(p) for p in coords])
        try:
            hull = SciPyConvexHull(coords_2d)
        except:
            if not silent:
                print("Topology.ConvexHull - Error computing 2D hull. Returning None.")
            return None

        # Reconstruct 3D points from 2D hull
        hull_indices = hull.vertices
        sorted_coords = coords[hull_indices]

        edges = []
        for i in range(len(sorted_coords)):
            p1 = sorted_coords[i]
            p2 = sorted_coords[(i + 1) % len(sorted_coords)]
            v1 = Vertex.ByCoordinates(_as_topology(*p1))
            v2 = Vertex.ByCoordinates(_as_topology(*p2))
            edges.append(Edge.ByVertices(_as_topology([v1, v2]), tolerance=tolerance))
        wire = Wire.ByEdges(_as_topology(edges), tolerance=tolerance)
        return Face.ByWire(wire, tolerance=tolerance)

    else:
        # Full 3D case
        try:
            hull = SciPyConvexHull(coords)
        except:
            try:
                hull = SciPyConvexHull(coords, qhull_options="QJ")
            except Exception as e:
                if not silent:
                    print(f"Topology.ConvexHull - 3D hull failed even with QJ: {e}")
                return None

        faces = []
        for simplex in hull.simplices:
            points = coords[simplex]
            edges = []
            for i in range(len(points)):
                p1 = points[i]
                p2 = points[(i + 1) % len(points)]
                v1 = Vertex.ByCoordinates(_as_topology(*p1))
                v2 = Vertex.ByCoordinates(_as_topology(*p2))
                edges.append(Edge.ByVertices(_as_topology([v1, v2]), tolerance=tolerance))
            wire = Wire.ByEdges(_as_topology(edges), tolerance=tolerance)
            face = Face.ByWire(wire, tolerance=tolerance)
            faces.append(face)

        convex_hull = Cell.ByFaces(faces, tolerance=tolerance, silent=True)
        if convex_hull:
            convex_hull_2 = Topology.RemoveCoplanarFaces(convex_hull, tolerance=tolerance, silent=True)
            if not convex_hull_2:
                return convex_hull
        else:
            convex_hull = Shell.ByFaces(faces, tolerance=tolerance, silent=True)
            if convex_hull:
                convex_hull_2 = Topology.RemoveCoplanarFaces(convex_hull, tolerance=tolerance)
                if not convex_hull_2:
                    return convex_hull
            else:
                convex_hull = Cluster.ByTopologies(_as_topology(faces))
        return convex_hull




In [ ]:
from topologicpy.Face import Face
from topologicpy.Cell import Cell
r = Cell.Cone()
r2 = Topology.Translate(_as_topology(r), 0.5, 0, 0)
r = Topology.Difference(r, r2)
ch = ConvexHull(r, silent=False)
print("Convex Hull:", ch)
Topology.Show(r)
Topology.Show(ch)
